## Misfit evaluation of CTSM model and Calibration

**This code assumes that the NEON_Simulation_Tutorial ran previosly, in order to obtain the settings of CTSM simulations**

In [ ]:
#Import Libraries
%matplotlib inline

import os
import sys
import time
import datetime

import numpy as np
import pandas as pd
import xarray as xr

from glob import glob
from os.path import join, expanduser

import matplotlib
import matplotlib.pyplot as plt

from scipy import stats

# Add project root to path so analytics_modules is importable
sys.path.insert(0, os.path.abspath('..'))

from analytics_modules import *

#from neon_utils import download_eval_files
#from model_misfit import *

## Preparation of Data

In [ ]:
#Specify the year below, the neon site should be the same downloaded previosly
year = "2018"

#Change the 4-character NEON site below.
neon_site= "ABBY"

import os
os.environ['site'] = neon_site

### Load CTSM model datafiles

In [ ]:
sim_path = "/home/user/archive/"+neon_site+".transient/lnd/hist/"
sim_files = sorted(glob(join(sim_path,neon_site+".transient.clm2.h1."+year+"*.nc")))

# This notebook expects a completed CTSM transient run for {neon_site}/{year}.
# If sim_files is empty, run the NEON simulation tutorial first or point
# sim_path at an archive directory you already have.
if not sim_files:
    raise FileNotFoundError(
        f"No CTSM history files at {sim_path}.\n"
        f"Run a transient case for {neon_site}/{year} first, e.g.:\n"
        f"  run_neon_v2 --neon-sites {neon_site} --output-root ~/CLM-NEON --no-batch"
    )
print("All Simulation files: [", len(sim_files), "files]")

In [ ]:
######## Manage the sim_files, as before Here we use the python function xarray.open_mfdataset, which opens multiple netcdf files as a single xarray dataset. For more information on this xarray function, visit the xarray website.

start = time.time()

ds_ctsm = xr.open_mfdataset(sim_files, decode_times=True, combine='by_coords')

end = time.time()
print("Reading all simulation files took:", end-start, "seconds.")

In [ ]:
### Vars of simulations
vareval = ['FCEV', 'FCTR', 'FGEV','GPP', 'H2OSOI']
df_ctsm_sim1 = ctsm_sim_depth(sim_files, vareval, levsoi=2.5)


df_ctsm_sim1

### Read Neon data

In [ ]:
##### download

eval_dir = "/home/user/evaluation_files/"

# TODO: `download_eval_files(neon_site, eval_dir, year)` is referenced here
# but no such function exists in analytics_modules. Either implement it (it
# should fetch NEON eval NetCDFs to eval_dir/<site>/), or place the files
# manually before running this cell.
# download_eval_files(neon_site, eval_dir, year)

### loading

eval_path = os.path.join('/home/user/evaluation_files/',neon_site)

eval_files = sorted(glob(join(eval_path,neon_site+"_eval_"+year+"*.nc")))
if not eval_files:
    raise FileNotFoundError(
        f"No NEON evaluation files at {eval_path}.\n"
        f"Either implement download_eval_files() or place {neon_site}_eval_{year}*.nc there manually."
    )

print("All Observation files:")
print(*eval_files,sep='\n')

start = time.time()

ds_eval = xr.open_mfdataset(eval_files, decode_times=True, combine='by_coords')

end = time.time()
print("Reading all observation files took:", end-start, "s.")

### Merging datasets

In [ ]:
# Convert the NEON observations xarray dataset to a flat DataFrame so we
# can attach simulated columns alongside the observed ones.
df_eval = ds_eval.to_dataframe().reset_index()

#-- make df_all that includes both obs and sim
df_all_gpp = df_eval

#-- add simulation data to df_all and adjust for offset time dimension:
for var in vareval:
    sim_var_name = "sim_"+var
    #-- shift simulation data by one
    df_all_gpp[sim_var_name]=df_ctsm_sim1[var].shift(-1).values
    
    
df_all_gpp

## Misfit evaluation and comparison

### Statistics

In [ ]:
########## step 1, call the misfit functions to evaluate model performance
var = 'GPP'
df = df_all_gpp

fig, residuals, metrics, conclusion = residuals_plots(
    df[f"{var}"],
    df[f"sim_{var}"],
    bins=40,
    savepath=None,
)
print(conclusion)
print(metrics)




### Kalman Filter

In [ ]:
######## step 2: as misfit happened, so then now its a KF
df_calib, m = calibrate_and_evaluate(df, var)
df_calib.columns



### Time series data

In [ ]:
label = f"{neon_site} Neon Site, {year}"

time_series_comparison(df_calib,label, var)